# Reward encoding in visual cortex: Zhong et al. 2025

**Question:** as mice learn a visual discrimination task, does the **proportion**
of reward-encoding neurons in each visual-cortical region change, and does their
**activation strength** change — tracked day by day, and compared against an
**unsupervised** cohort that sees identical stimuli but is never rewarded?

**Method:** for each sampled neuron, split its rewarded-corridor trials by
whether the sound cue (whose position tracks the reward zone) fell **early** or
**late**, and compute `d' = 2*(mean_late - mean_early) / (std_late + std_early)`
in the reward zone — Zhong et al.'s own reward-prediction index (their Fig. 4f–g).
The flag is **cross-validated**: a neuron counts only if `|d'| >= 0.3` reproduces
across two interleaved trial folds with the same sign, so read noise doesn't pass.
We use this cue-based index instead of an encoding-model reward *ablation* because
the cue plays whether or not water is delivered — so, unlike an ablation of the
water-delivery regressor (which we tried first; see `docs/method.md` for why it
was retired), the **same index is defined for unsupervised mice too**, making the
supervised-vs-unsupervised contrast a fair one rather than zero-by-construction.

Per (cohort, stage, region, mouse) we get two numbers: the **percentage** of
sampled neurons flagged (recruitment) and their **mean cross-validated |d'|**
(gain). Both are tracked across four stages — `train1_before/after`,
`train2_before/after` — and each **animal** imaged in every stage of its cohort
contributes one connected trajectory, so the final figure shows individual
animals, not just group means.

**Run in Colab:** **Runtime → Run all**. No Drive mount needed; files stream from
the official Figshare deposit and are cached (one SVD + retinotopy file per
mouse, ~100–180 MB each). With the animal counts found in the deposit — 4
supervised, 6 unsupervised, each imaged across all four stages — a full run
downloads ~2 GB and completes in well under an hour on a standard Colab runtime.


## 1. Setup and configuration

Colab already includes the scientific Python packages used below. The configuration
cell is the only place to change sample size, threshold, or random seed.


In [ ]:
from __future__ import annotations

import platform
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import seaborn as sns
try:
    from tqdm.auto import tqdm  # progress bars with ETA (preinstalled in Colab)
except ModuleNotFoundError:  # ponytail: minimal test env lacks tqdm; no-op fallback
    def tqdm(iterable=None, **_):
        return iterable

sns.set_theme(style="whitegrid", context="notebook")
np.set_printoptions(precision=4, suppress=True)


In [ ]:
SEED = 42
DATA_DIR = Path("/content/Zhong_et_al_2025")
OUTPUT_DIR = Path("/content")
ARTICLE_ID = 28811129        # the whole Figshare deposit

AREA_CODES = {
    "V1": (8,),
    "mHV": (0, 1, 2, 9),
    "lHV": (5, 6),
    "aHV": (3, 4),
}

N_PER_REGION = 250           # neurons sampled per region per mouse (capped at available)
DPRIME_THRESHOLD = 0.3       # Zhong et al.'s reward-prediction cutoff, |d'| >= 0.3
N_SHUFFLES = 100              # late/early label shuffles -> empirical chance floor
REWARD_ZONE_DM = (5, 40)     # corridor dm averaged for reward-zone activity (0.5-4 m)

# The learning-time axis: two training blocks, before vs after learning in each.
STAGE_ORDER = ["train1_before", "train1_after", "train2_before", "train2_after"]

# (cohort, stage) -> behavior filename. Both cohorts get the SAME cue-based d'
# index: the sound cue plays whether or not water is delivered. Confirmed on an
# unsupervised file (TX88_2022_06_20_1): RewardFr all-NaN, isRew all-False, but
# SoundPos present with the same range as supervised sessions - a genuine,
# non-degenerate control, unlike the retired reward-ablation approach (which
# ablated water delivery and so scored unsupervised mice zero by construction).
COHORT_STAGE_FILES = {
    ("supervised", "train1_before"): "Beh_sup_train1_before_learning.npy",
    ("supervised", "train1_after"): "Beh_sup_train1_after_learning.npy",
    ("supervised", "train2_before"): "Beh_sup_train2_before_learning.npy",
    ("supervised", "train2_after"): "Beh_sup_train2_after_learning.npy",
    ("unsupervised", "train1_before"): "Beh_unsup_train1_before_learning.npy",
    ("unsupervised", "train1_after"): "Beh_unsup_train1_after_learning.npy",
    ("unsupervised", "train2_before"): "Beh_unsup_train2_before_learning.npy",
    ("unsupervised", "train2_after"): "Beh_unsup_train2_after_learning.npy",
}

print(f"Python {platform.python_version()} | NumPy {np.__version__}")
print(f"Seed={SEED}; {N_PER_REGION} neurons/region; |d'| >= {DPRIME_THRESHOLD}; "
      f"{len(COHORT_STAGE_FILES)} cohort x stage sessions")


## 2. Data access

Files are streamed from the official Figshare deposit and resolved by name against
a one-time manifest fetch, then cached and size-verified: a truncated download
fails loudly instead of producing quiet nonsense. Zhong's SVD files contain an
unused serialized scikit-learn object; this notebook reads only `U` and `V`.


In [ ]:
# --- Data access: resolve any deposit file by name, cached and size-verified ---
def fetch_manifest(article_id=ARTICLE_ID):
    """Map every filename in the Figshare deposit to its id and size (one request)."""
    r = requests.get(f"https://api.figshare.com/v2/articles/{article_id}", timeout=(15, 60))
    r.raise_for_status()
    return {f["name"]: {"id": f["id"], "size": f["size"]} for f in r.json()["files"]}


def download_by_name(name, manifest, data_dir=DATA_DIR):
    """Download one deposit file by exact name; cached and size-verified."""
    data_dir = Path(data_dir); data_dir.mkdir(parents=True, exist_ok=True)
    if name not in manifest:
        raise KeyError(f"{name} is not in the Figshare deposit")
    target = data_dir / name
    expected = manifest[name]["size"]
    if target.exists() and target.stat().st_size == expected:
        return target
    url = f"https://ndownloader.figshare.com/files/{manifest[name]['id']}"
    with requests.get(url, stream=True, timeout=(15, 600)) as r:
        r.raise_for_status()
        with target.open("wb") as fh:
            for chunk in r.iter_content(1024 * 1024):
                fh.write(chunk)
    if target.stat().st_size != expected:
        raise RuntimeError(f"Truncated download for {name}; rerun the cell to retry.")
    return target


## 3. Region mapping and balanced neuron sampling

Region membership comes from the retinotopy file. We sample up to `N_PER_REGION`
neurons from each mapped region with a fixed seed, independently of activity or
model fit, then reconstruct only those neurons from Zhong's 400-component SVD.


In [ ]:
def map_regions(iarea):
    """Map Zhong's numeric area codes onto the four regions we compare."""
    iarea = np.asarray(iarea)
    labels = np.full(len(iarea), "unmapped", dtype=object)
    for name, codes in AREA_CODES.items():
        labels[np.isin(iarea, codes)] = name
    return labels.astype(str)


def balanced_sample(regions, n_per_region, seed=42):
    """Pick up to n_per_region neurons per region, blind to activity and model fit.

    If a region has fewer neurons than requested we take all of them (and say so)
    instead of crashing, so the same call works on tiny and huge sessions alike.
    Regions can therefore end up with different counts - that is fine, because we
    always report *percentages* with a per-region denominator downstream.
    """
    regions = np.asarray(regions)
    rng = np.random.default_rng(seed)
    selected, labels = [], []
    for region in AREA_CODES:
        candidates = np.flatnonzero(regions == region)
        n = min(n_per_region, len(candidates))
        if n == 0:
            print(f"  {region}: no mapped neurons, skipping")
            continue
        if n < n_per_region:
            print(f"  {region}: only {n} neurons available (< {n_per_region} requested)")
        selected.append(np.sort(rng.choice(candidates, n, replace=False)))
        labels.extend([region] * n)
    return np.concatenate(selected), np.asarray(labels)


def reconstruct_selected(svd, selected):
    """Rebuild activity for just the sampled neurons from the 400-component SVD."""
    U = np.asarray(svd["U"], dtype=np.float32)  # components x neurons
    V = np.asarray(svd["V"], dtype=np.float32)  # components x frames
    return (U[:, np.asarray(selected, dtype=int)].T @ V).T.astype(np.float32, copy=False)


## 4. Cross-validated d′(late vs early cue)

Split each rewarded-corridor trial's reward-zone activity by whether the sound
cue fell early or late, and ask how different those two trial groups look
(Zhong et al.'s Fig. 4f–g index). A single `|d'| >= 0.3` on all trials is
deceptively noisy: with only a few dozen trials, pure noise clears 0.3 often. So
we **cross-validate** — split into two interleaved folds (first sorted by the
late/early label so each fold gets a balanced mix), score each independently,
and flag a neuron only if both folds agree in magnitude and sign. Its
**strength** (the activation tracked across days) is the cross-validated `|d'|`
magnitude itself, reported for *all* sampled neurons so it is not biased by
the selection.


In [ ]:
def _dprime_between(trial_activity, group):
    """Signal-detection d' between two groups of trials, per neuron.

    trial_activity: neurons x trials. group: bool over trials (True = group A).
    d' = 2*(mean_A - mean_B) / (std_A + std_B) - the same form used in the paper
    and the 50k loading notebook. The tiny +1e-9 just avoids a divide-by-zero for
    a silent neuron.
    """
    a = trial_activity[:, group]
    b = trial_activity[:, ~group]
    return 2 * (a.mean(1) - b.mean(1)) / (a.std(1) + b.std(1) + 1e-9)


def _reward_encoding(trial_mean, late, threshold):
    """Cross-validated reward encoding per neuron. Returns (flag, strength).

    - flag (for the proportion): True only if |d'(late vs early)| >= threshold in
      both folds with the same sign - a real anticipatory effect reproduces across
      folds; noise almost never does.
    - strength (for the activation): the neuron's cross-validated reward-d' magnitude,
      (|d'_foldA| + |d'_foldB|)/2 - reported for all sampled neurons (not just the
      flagged ones, so it is not biased by the selection).
    """
    order = np.argsort(late)                      # early trials first, then late
    tm, lt = trial_mean[:, order], late[order]
    fold = np.arange(tm.shape[1]) % 2             # interleave -> both folds balanced
    a, b = fold == 0, fold == 1
    d_a = _dprime_between(tm[:, a], lt[a])
    d_b = _dprime_between(tm[:, b], lt[b])
    flag = (np.abs(d_a) >= threshold) & (np.abs(d_b) >= threshold) & (np.sign(d_a) == np.sign(d_b))
    strength = (np.abs(d_a) + np.abs(d_b)) / 2
    return flag, strength


def reward_dprime_session(beh, mouse, manifest,
                          n_per_region=N_PER_REGION,
                          reward_zone=REWARD_ZONE_DM, seed=SEED):
    """Reward encoding per sampled neuron for one mouse, grouped by region.

    Split the rewarded-corridor trials by where the sound cue fell (early vs late),
    average each neuron's activity in the reward zone per trial, and ask how
    different late-cue vs early-cue trials look. `_reward_encoding` turns that into
    a cross-validated flag and a strength (reward-d' magnitude = its "activation").
    Works identically for supervised and unsupervised mice: the cue and reward
    zone exist in both, water delivery does not enter this computation at all.
    """
    stim_id, uniqW, wall_name = beh["stim_id"], beh["UniqWalls"], beh["WallName"]
    if not np.any(stim_id == 2):
        raise ValueError("no rewarded stimulus (leaf1) in this session")
    rewarded_wall = uniqW[stim_id == 2][0]                    # leaf1 = rewarded stimulus
    trial_is_rewarded = wall_name == rewarded_wall
    cue_pos = beh["SoundPos"] if "SoundPos" in beh else np.mod(beh["SoundDelPos"], 60)
    cue_pos = np.asarray(cue_pos, dtype=float)
    trial_frame = np.asarray(beh["ft_trInd"], dtype=float)    # trial index of each frame
    pos_frame = np.asarray(beh["ft_Pos"], dtype=float)        # position in corridor (dm)
    wall_frame = np.asarray(beh["ft_WallID"])                 # stimulus shown each frame
    moving = np.asarray(beh["ft_move"]) > 0

    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", message="Trying to unpickle estimator")
        svd = np.load(download_by_name(f"{mouse}_SVD_dec.npy", manifest),
                      allow_pickle=True).item()
    retin = np.load(download_by_name(f"{mouse[:-1]}trans.npz", manifest), allow_pickle=False)
    sel, sel_regions = balanced_sample(map_regions(retin["iarea"]), n_per_region, seed=seed)
    activity = reconstruct_selected(svd, sel)                 # frames x neurons
    del svd

    # per-trial reward-zone mean activity: neurons x rewarded-trials
    n_frames = min(activity.shape[0], len(trial_frame))
    activity = activity[:n_frames]
    in_zone = ((wall_frame[:n_frames] == rewarded_wall)
               & (pos_frame[:n_frames] >= reward_zone[0])
               & (pos_frame[:n_frames] < reward_zone[1])
               & moving[:n_frames])
    rewarded_trials = np.flatnonzero(trial_is_rewarded)
    frame_trial = np.rint(trial_frame[:n_frames])
    trial_mean = np.full((activity.shape[1], len(rewarded_trials)), np.nan, np.float32)
    for j, t in enumerate(rewarded_trials):
        fr = in_zone & (frame_trial == t)
        if fr.any():
            trial_mean[:, j] = activity[fr].mean(0)

    keep = ~np.isnan(trial_mean).any(0)                       # trials with reward-zone frames
    trial_mean = trial_mean[:, keep]
    cue = cue_pos[rewarded_trials][keep]
    if np.ptp(cue) == 0:
        raise ValueError("cue position never varies - cannot split early vs late")
    late = cue > np.median(cue)                               # late-cue vs early-cue trials
    if late.sum() < 8 or (~late).sum() < 8:                   # need enough for a 2-fold split
        raise ValueError(f"too few early/late-cue trials for cross-validated d' "
                         f"({(~late).sum()} early / {late.sum()} late)")

    flag, strength = _reward_encoding(trial_mean, late, DPRIME_THRESHOLD)
    return {
        "flag_by_region": {r: flag[sel_regions == r] for r in AREA_CODES},
        "strength_by_region": {r: strength[sel_regions == r] for r in AREA_CODES},
        "trial_mean": trial_mean, "late": late,
    }


def run_dprime_synthetic_check():
    """One ramping neuron + noise neurons; the flag must catch the ramp, not the noise."""
    rng = np.random.default_rng(3)
    late = np.arange(60) % 2 == 0
    tm = rng.normal(0, 1.0, (6, 60)).astype(np.float32)
    tm[0, late] += 3.0                       # neuron 0 ramps on late-cue trials
    flag, strength = _reward_encoding(tm, late, DPRIME_THRESHOLD)
    if not (flag[0] and not flag[1:].any()):
        raise AssertionError(f"d' synthetic check failed: flag={flag}")
    return {"flagged": int(flag.sum()), "ramp_strength": round(float(strength[0]), 2)}


## 5. Run across cohorts, stages, and animals

An **animal**, not a recording session, is the replicate: the same mouse is
imaged on a different date for "before" and "after" within a training block, so
we key by the short animal id (`mouse.split("_")[0]`) and only that id needs to
appear in every stage of a cohort for its trajectory to be unbroken. Any animal
missing a stage (too few reward events, no usable early/late split, or simply
absent from that stage's file) just leaves a gap in its own line — it is not
dropped from the others.


In [ ]:
# --- Run cross-validated d' across every (cohort, stage), animal-paired ---
# One SVD + retinotopy file per animal per stage (~100-180 MB, cached after the
# first run). Downloads happen once; rerunning this cell reuses the cache.
print("d' sanity check:", run_dprime_synthetic_check())
manifest = fetch_manifest()


def animal_id(session_key):
    return session_key.split("_")[0]


stage_beh = {
    key: np.load(download_by_name(fname, manifest), allow_pickle=True).item()
    for key, fname in COHORT_STAGE_FILES.items()
}

# Only animals imaged in EVERY stage of a cohort (and with SVD + retinotopy
# files available) give an unbroken before/after trajectory across both
# training blocks.
cohorts = sorted({cohort for cohort, _ in COHORT_STAGE_FILES})
paired_animals = {}
for cohort in cohorts:
    per_stage_ids = [
        {animal_id(m) for m in stage_beh[(cohort, stage)]
         if f"{m}_SVD_dec.npy" in manifest and f"{m[:-1]}trans.npz" in manifest}
        for stage in STAGE_ORDER
    ]
    paired_animals[cohort] = sorted(set.intersection(*per_stage_ids))
    print(f"{cohort}: {len(paired_animals[cohort])} animal(s) paired across all "
          f"4 stages -> {paired_animals[cohort]}")

rows, floor_rows, skipped = [], [], []
rng = np.random.default_rng(SEED)
for cohort in cohorts:
    for stage in STAGE_ORDER:
        beh_all = stage_beh[(cohort, stage)]
        animals = paired_animals[cohort]
        for animal in tqdm(animals, desc=f"{cohort}/{stage}", unit="mouse"):
            mouse = next(m for m in beh_all if animal_id(m) == animal)
            try:
                res = reward_dprime_session(beh_all[mouse], mouse, manifest)
            except (KeyError, ValueError, RuntimeError) as err:
                skipped.append((cohort, stage, animal, str(err)))
                continue
            for region in AREA_CODES:
                flag = res["flag_by_region"][region]
                strength = res["strength_by_region"][region]
                if len(flag):
                    rows.append({
                        "cohort": cohort, "stage": stage, "region": region,
                        "animal": animal, "mouse": mouse,
                        "pct_reward": 100 * np.mean(flag),          # how MANY (recruitment)
                        "mean_dprime": float(np.mean(strength)),    # how STRONGLY (gain)
                        "n_neurons": len(flag),
                    })
            # empirical chance floor: same cross-validated flag on shuffled late/early labels
            tm, late = res["trial_mean"], res["late"]
            shuffled_pct = [
                100 * np.mean(_reward_encoding(tm, rng.permutation(late), DPRIME_THRESHOLD)[0])
                for _ in range(N_SHUFFLES)
            ]
            floor_rows.append({"cohort": cohort, "pct": float(np.mean(shuffled_pct))})

results_df = pd.DataFrame(rows)
chance_floor = (pd.DataFrame(floor_rows).groupby("cohort")["pct"].mean()
               if floor_rows else pd.Series(dtype=float))
n_animals = results_df[["cohort", "animal"]].drop_duplicates().shape[0] if len(results_df) else 0
print(f"\n{len(results_df)} region x animal x stage rows from {n_animals} animal-cohort pairs")
print(f"chance floor (label-shuffle) by cohort:\n{chance_floor}")
if skipped:
    print(f"{len(skipped)} (cohort, stage, animal) skipped - inspect `skipped` for reasons.")
if len(results_df):
    display(results_df.groupby(["cohort", "stage", "region"])[["pct_reward", "mean_dprime"]]
            .mean().round(3))


## 6. Results — proportion and activation across learning, by region and cohort

Top row = **recruitment** (percentage of neurons flagged as reward-encoding).
Bottom row = **gain** (their mean cross-validated `|d'|`). In each panel, thin
lines are individual animals (a gap means that animal's stage was skipped), the
box shows the distribution across animals at that stage, and the bold dot +
line is the cohort mean. Supervised and unsupervised are overlaid so the
paper's key dissociation — the reward-prediction signal should be
supervised-only — is visible directly in the recruitment/gain trajectories,
not just inferred from a separate contrast.


In [ ]:
# --- Figure: recruitment (top) and gain (bottom) across learning, by region ---
COHORT_COLORS = {"supervised": "#4C72B0", "unsupervised": "#DD8452"}
COHORT_OFFSET = {"supervised": -0.12, "unsupervised": 0.12}


def _panel(ax, metric, region, show_floor):
    xpos = {s: i for i, s in enumerate(STAGE_ORDER)}
    for cohort in cohorts:
        color = COHORT_COLORS.get(cohort, "0.3")
        offset = COHORT_OFFSET.get(cohort, 0.0)
        sub = results_df[(results_df.region == region) & (results_df.cohort == cohort)]
        if not len(sub):
            continue
        for animal, grp in sub.groupby("animal"):          # per-animal trajectory (gaps = nan)
            by_stage = grp.set_index("stage")[metric]
            ys = [by_stage.get(s, np.nan) for s in STAGE_ORDER]
            xs = [xpos[s] + offset for s in STAGE_ORDER]
            ax.plot(xs, ys, color=color, lw=0.8, alpha=0.35, zorder=1)
        box_data, box_pos = [], []                          # distribution across animals
        for s in STAGE_ORDER:
            vals = sub.loc[sub.stage == s, metric].to_numpy()
            if len(vals):
                box_data.append(vals); box_pos.append(xpos[s] + offset)
        if box_data:
            bp = ax.boxplot(box_data, positions=box_pos, widths=0.18, patch_artist=True,
                            showfliers=False, zorder=2)
            for patch in bp["boxes"]:
                patch.set_facecolor(color); patch.set_alpha(0.25); patch.set_edgecolor(color)
            for element in ("whiskers", "caps", "medians"):
                for line in bp[element]:
                    line.set_color(color)
        means, xs_mean = [], []                             # bold cohort mean +/- s.e.m.
        for s in STAGE_ORDER:
            vals = sub.loc[sub.stage == s, metric].to_numpy()
            if not len(vals):
                continue
            x = xpos[s] + offset
            sem = vals.std(ddof=1) / np.sqrt(len(vals)) if len(vals) > 1 else 0.0
            ax.errorbar(x, vals.mean(), yerr=sem, fmt="o", color=color, markersize=5,
                       lw=2, capsize=3, zorder=4)
            means.append(vals.mean()); xs_mean.append(x)
        if len(xs_mean) > 1:
            ax.plot(xs_mean, means, color=color, lw=2, zorder=3)
        if show_floor and cohort in chance_floor.index:
            ax.axhline(chance_floor[cohort], ls="--", color=color, lw=1, alpha=0.6)
    ax.set_xticks(range(len(STAGE_ORDER)))
    ax.set_xticklabels([s.replace("_", "\n") for s in STAGE_ORDER], fontsize=8)
    ax.set_xlim(-0.5, len(STAGE_ORDER) - 0.5)
    ax.spines[["top", "right"]].set_visible(False)


if len(results_df) == 0:
    print("No sessions produced results - check `skipped`.")
else:
    fig, axes = plt.subplots(2, 4, figsize=(17, 8), sharex=True)
    metrics = [("pct_reward", "% reward-encoding neurons\n(cross-val d', |d'|>=0.3)", True),
               ("mean_dprime", "mean |d'| (late vs early cue)\n(activation strength)", False)]
    for row, (metric, ylab, show_floor) in enumerate(metrics):
        for col, region in enumerate(AREA_CODES):
            _panel(axes[row, col], metric, region, show_floor)
            if row == 0:
                axes[row, col].set_title(region, fontweight="bold")
        axes[row, 0].set_ylabel(ylab)
    handles = [plt.Line2D([], [], color=COHORT_COLORS["supervised"], marker="o", label="supervised"),
              plt.Line2D([], [], color=COHORT_COLORS["unsupervised"], marker="o", label="unsupervised"),
              plt.Line2D([], [], color="0.4", ls="--", label="chance floor (label shuffle)")]
    axes[0, -1].legend(handles=handles, fontsize=8, loc="upper left", frameon=False)
    fig.suptitle("Reward-encoding neurons across learning - recruitment (top) and gain (bottom)\n"
                "by region; thin lines = individual animals, box = distribution across animals, "
                "bold = cohort mean", y=1.03)
    plt.tight_layout(); plt.show()

    csv_path = OUTPUT_DIR / "reward_encoding_by_animal.csv"
    results_df.to_csv(csv_path, index=False)
    print(f"Saved {csv_path} ({len(results_df)} rows)")
    try:
        from google.colab import files as colab_files
        colab_files.download(str(csv_path))
    except ImportError:
        pass


## Interpretation checklist

- **Recruitment vs gain.** If the proportion of flagged neurons rises with
  learning while their mean `|d'|` stays flat, that's *recruitment* (new
  neurons join the code). If the proportion is flat but `|d'|` rises, that's
  *gain* (the same-sized population responds more strongly). Both can be true
  at once, and region/cohort can differ.
- **Supervised vs unsupervised is the paper's headline test.** Zhong et al.
  report the reward-prediction signal in anterior HVAs only with reward
  (task p=0.0069 vs unsupervised p=0.708, their Fig. 4g). Support for that
  account here looks like: the supervised line sits clearly above its chance
  floor and rises with learning in `aHV`, while the unsupervised line tracks
  its own chance floor throughout.
- **No individual-neuron tracking.** "Before" and "after" are different
  imaging sessions with independent neuron indexing (confirmed against
  `Imaging_Exp_info.npy` — no cell-registration file in the deposit), so a
  rising proportion or `|d'|` reflects the *population*, not the same cells
  strengthening. An animal's trajectory is a paired population-level
  comparison, not a tracked-neuron one.
- **Chance floor, not p<0.05 alone.** The floor is an empirical shuffle of the
  late/early label through the *same* cross-validated flag, per cohort — read
  a line against its own floor, not zero.
- **No multiple-comparison correction** across regions or stages; treat a
  single borderline crossing as "worth a follow-up," not proof.
